# Pharmaceutical Document RAG — Google Colab

A self-contained, end-to-end RAG pipeline for pharmaceutical PDF question-answering.

| Component | Details |
|---|---|
| **Embedding** | `sentence-transformers/all-MiniLM-L6-v2` (GPU-accelerated) |
| **Chunking** | Fixed-size (512 tokens, 50 overlap) |
| **Retrieval** | Hybrid Vector + BM25, reciprocal rerank (top-5) |
| **LLM** | Mistral 7B Instruct (GGUF, local) |
| **OCR** | Tesseract (parallel, 200 DPI) |
| **UI** | Gradio (public share link) |

## How to run
1. **Runtime → Change runtime type → T4 GPU** *(recommended for speed)*
2. Run all cells top-to-bottom (`Runtime → Run all`)
3. Step 4 downloads ~4 GB — wait for the model to finish before the next cell
4. The Gradio cell prints a public share URL — open it in any browser


In [ ]:
# ── Step 1: System dependencies ──────────────────────────────────────────────
# Installs Tesseract OCR (needed for scanned / image-based PDF pages).
import subprocess

result = subprocess.run(
    ["apt-get", "install", "-y", "-q", "tesseract-ocr", "poppler-utils"],
    capture_output=True, text=True,
)
if result.returncode == 0:
    print("Tesseract OCR installed.")
else:
    print("apt-get stderr:", result.stderr[:400])


In [ ]:
# ── Step 2: Python packages ───────────────────────────────────────────────────
%pip install -q pymupdf
%pip install -q "llama-index>=0.14.15" llama-index-core
%pip install -q llama-index-embeddings-huggingface
%pip install -q llama-index-llms-llama-cpp
%pip install -q llama-index-retrievers-bm25
%pip install -q sentence-transformers huggingface-hub
%pip install -q pytesseract pillow
%pip install -q "gradio>=4.0.0"


In [ ]:
# ── Step 3: Install llama-cpp-python (GPU-aware) ─────────────────────────────
# Tries to install a pre-built CUDA 12.2 wheel first (for Colab T4 GPU).
# Falls back to the CPU-only wheel if CUDA is unavailable or the wheel fails.
import subprocess, sys

try:
    import torch
    has_cuda = torch.cuda.is_available()
except ImportError:
    has_cuda = False

if has_cuda:
    import torch
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
    print("Installing llama-cpp-python with CUDA 12.2 support...")
    r = subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "llama-cpp-python",
            "--extra-index-url",
            "https://abetlen.github.io/llama-cpp-python/whl/cu122",
            "-q",
        ],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print("Pre-built CUDA wheel unavailable — falling back to CPU build.")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "llama-cpp-python", "-q"],
            check=True,
        )
    else:
        print("llama-cpp-python (CUDA) installed.")
else:
    print("No GPU detected — installing CPU-only llama-cpp-python...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "llama-cpp-python", "-q"],
        check=True,
    )
    print("llama-cpp-python (CPU) installed.")


In [ ]:
# ── Step 4: Download Mistral-7B-Instruct-v0.2 GGUF (~4 GB) ──────────────────
# Downloaded once into /content/models. Colab may cache it across sessions
# if Google Drive is mounted; otherwise it re-downloads on each new runtime.
import os
from huggingface_hub import hf_hub_download

os.makedirs("/content/models", exist_ok=True)

print("Downloading Mistral-7B-Instruct-v0.2 Q4_K_M (~4 GB). This may take a few minutes...")
MODEL_PATH = hf_hub_download(
    repo_id="TheBloke/Mistral-7B-Instruct-v0.2-GGUF",
    filename="mistral-7b-instruct-v0.2.Q4_K_M.gguf",
    local_dir="/content/models",
)
print(f"Model ready: {MODEL_PATH}")


In [ ]:
# ── Step 5: Imports + RAGPipeline class ───────────────────────────────────────
import logging
import os
import shutil
from concurrent.futures import ThreadPoolExecutor
from typing import Any, Dict, Iterator, List, Optional


import fitz  # PyMuPDF
import torch
from llama_index.core import (
    Document, Settings, StorageContext, VectorStoreIndex, load_index_from_storage,
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.prompts import PromptTemplate
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import QueryFusionRetriever, VectorIndexRetriever
from llama_index.core.vector_stores.types import MetadataFilter, MetadataFilters
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.llama_cpp import LlamaCPP
from llama_index.retrievers.bm25 import BM25Retriever

try:
    import pytesseract
    from PIL import Image as PILImage
    _OCR_AVAILABLE = True
except ImportError:
    pytesseract = None
    PILImage = None
    _OCR_AVAILABLE = False


def _resolve_tesseract_cmd() -> Optional[str]:
    if not _OCR_AVAILABLE:
        return None
    if pytesseract is None:
        return "tesseract"
    configured = str(getattr(pytesseract.pytesseract, "tesseract_cmd", "") or "").strip()
    if configured:
        found = shutil.which(configured)
        if found:
            return found
    return shutil.which("tesseract")


def _is_ocr_runtime_available() -> bool:
    if not _OCR_AVAILABLE:
        return False
    cmd = _resolve_tesseract_cmd()
    if not cmd:
        return False
    if pytesseract is not None:
        pytesseract.pytesseract.tesseract_cmd = cmd
    return True


logging.basicConfig(level="INFO")
logger = logging.getLogger(__name__)

_PHARMA_QA_PROMPT = PromptTemplate(
    "You are a pharmaceutical document assistant. "
    "Answer the question based ONLY on the context provided below. "
    "Be as brief as possible: 1-2 sentences maximum. "
    "If a list is requested, use bullet points with no extra explanation. "
    "Never add background, reasoning, or context beyond what directly answers the question. "
    "Do not include citations in the answer text.\n\n"
    "Context:\n{context_str}\n\n"
    "Question: {query_str}\n\n"
    "Answer:"
)

_SCANNED_PAGE_CHAR_THRESHOLD = 100
_OCR_DPI = 200
_DOC_CLASSIFY_PROMPT_CHARS = 600

_PHARMA_DOC_CATEGORIES: List[str] = [
    "cover_letter", "certificate_of_quality", "packaging_specification",
    "bse_tse_declaration", "material_description", "supplier_qualification",
    "chain_of_custody", "unknown",
]

_KEYWORD_MAP: Dict[str, List[str]] = {
    "cover_letter": ["cover letter", "dear sir", "dear madam", "dear supplier",
                     "please find enclosed", "we herewith", "herewith enclosed"],
    "certificate_of_quality": ["certificate of quality", "certificate of analysis",
                                "cert. of quality", "coa ", "c.o.a"],
    "packaging_specification": ["packaging specification", "packaging spec",
                                 "pack spec", "label specification", "labelling specification"],
    "bse_tse_declaration": ["bse", "tse", "transmissible spongiform",
                             "bovine spongiform", "spongiform encephalopathy"],
    "material_description": ["material description", "material data sheet",
                              "product description", "substance description",
                              "raw material description"],
    "supplier_qualification": ["supplier qualification", "vendor qualification",
                                "approved supplier", "audit report", "supplier audit"],
    "chain_of_custody": ["chain of custody", "chain-of-custody", "custody transfer"],
}

_FEW_SHOT_EXAMPLES: str = (
    "cover_letter\n"
    "Example: \"Dear Supplier, Please find enclosed the updated documentation for batch 2024-001.\"\n\n"
    "certificate_of_quality\n"
    "Example: \"Certificate of Quality - Batch No: 12345 - Product: Excipient X - Conforms to specification.\"\n\n"
    "packaging_specification\n"
    "Example: \"Packaging Specification Rev. 3 - Primary container: HDPE bottle 250 mL - Closure torque: 15-20 Nm.\"\n\n"
    "bse_tse_declaration\n"
    "Example: \"BSE/TSE Declaration - We confirm that no materials of bovine or ovine origin are used.\"\n\n"
    "material_description\n"
    "Example: \"Material Description - Chemical name: Microcrystalline Cellulose - CAS: 9004-34-6 - Function: Filler.\"\n\n"
    "supplier_qualification\n"
    "Example: \"Supplier Qualification Report - Audit date: 2023-05 - Site: Plant A - Status: Approved.\"\n\n"
    "chain_of_custody\n"
    "Example: \"Chain of Custody - Transferred from Manufacturer X to Distributor Y on 2024-03-01.\"\n\n"
)


class RAGPipeline:
    """End-to-end RAG pipeline for pharmaceutical PDF question-answering."""

    def __init__(
        self,
        model_path: Optional[str] = None,
        embed_model_name: Optional[str] = None,
        similarity_top_k: Optional[int] = None,
        num_queries: int = 1,
        n_gpu_layers: Optional[int] = None,
        persist_dir: Optional[str] = None,
    ) -> None:
        _model_path: str = model_path or "/content/models/mistral-7b-instruct-v0.2.Q4_K_M.gguf"
        _embed_model: str = embed_model_name or "sentence-transformers/all-MiniLM-L6-v2"
        _top_k: int = similarity_top_k if similarity_top_k is not None else 5
        _gpu_layers: int = (
            n_gpu_layers if n_gpu_layers is not None
            else (-1 if torch.cuda.is_available() else 0)
        )

        if not os.path.exists(_model_path):
            raise FileNotFoundError(f"GGUF model not found: {_model_path}")

        self.similarity_top_k: int = _top_k
        self.num_queries: int = num_queries

        self.llm: LlamaCPP = LlamaCPP(
            model_path=_model_path,
            temperature=0.1,
            max_new_tokens=512,
            context_window=8192,
            model_kwargs={"n_gpu_layers": _gpu_layers},
            verbose=False,
        )

        self.embed_model: HuggingFaceEmbedding = HuggingFaceEmbedding(
            model_name=_embed_model,
            device="cuda" if torch.cuda.is_available() else "cpu",
        )

        Settings.embed_model = self.embed_model
        Settings.llm = self.llm

        self._splitter: SentenceSplitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
        self.persist_dir: Optional[str] = persist_dir
        self._chunks: List[Any] = []
        self._vector_index: Optional[VectorStoreIndex] = None
        self._query_engine: Optional[RetrieverQueryEngine] = None
        self._pdf_path: Optional[str] = None
        self._docs_classified: bool = False
        self._query_cache: Dict[tuple, Dict[str, Any]] = {}

    # ── PDF Loading ────────────────────────────────────────────────────────────

    def _ocr_page(self, page: Any) -> str:
        if pytesseract is None or PILImage is None:
            return ""
        pix = page.get_pixmap(dpi=_OCR_DPI)
        img = PILImage.frombytes("RGB", [pix.width, pix.height], pix.samples)
        return pytesseract.image_to_string(img)

    def load_pdf(self, pdf_path: str) -> List[Document]:
        documents: List[Document] = []
        file_name = os.path.basename(pdf_path)
        ocr_enabled = _is_ocr_runtime_available()

        with fitz.open(pdf_path) as doc:
            page_data = []
            for i, page in enumerate(doc):
                text = page.get_text()
                page_data.append((i, page, text))

            ocr_dict: Dict[int, str] = {}
            if ocr_enabled:
                pages_needing_ocr = [
                    (i, page) for i, page, text in page_data
                    if len(text.strip()) < _SCANNED_PAGE_CHAR_THRESHOLD
                ]
                if pages_needing_ocr:
                    with ThreadPoolExecutor(max_workers=min(4, os.cpu_count() or 2)) as executor:
                        ocr_results = list(
                            executor.map(lambda p: (p[0], self._ocr_page(p[1])), pages_needing_ocr)
                        )
                    ocr_dict = dict(ocr_results)

            for i, page, text in page_data:
                ocr_used = False
                if i in ocr_dict:
                    text = ocr_dict[i]
                    ocr_used = True
                if not text.strip():
                    continue
                documents.append(Document(
                    text=text,
                    metadata={
                        "file_name": file_name,
                        "page_number": i + 1,
                        "total_pages": len(doc),
                        "doc_type": "scanned" if ocr_used else "digital",
                        "ocr_used": ocr_used,
                        "source_id": f"{file_name}:p{i + 1}",
                    },
                ))

        logger.info("Loaded '%s': %d pages with content.", file_name, len(documents))
        return documents

    # ── Chunking ───────────────────────────────────────────────────────────────

    def _chunk(self, documents: List[Document]) -> List[Any]:
        if not documents:
            raise ValueError("No documents provided for chunking.")
        chunks = self._splitter.get_nodes_from_documents(documents)
        logger.info("Total chunks created: %d", len(chunks))
        return chunks

    # ── Indexing ───────────────────────────────────────────────────────────────

    def _index(self, chunks: List[Any]) -> VectorStoreIndex:
        vector_index = VectorStoreIndex.from_documents(chunks)
        if self.persist_dir:
            vector_index.storage_context.persist(persist_dir=self.persist_dir)
        return vector_index

    # ── Retriever ──────────────────────────────────────────────────────────────

    def _build_retriever(
        self, vector_index: VectorStoreIndex, chunks: List[Any]
    ) -> QueryFusionRetriever:
        vector_retriever = VectorIndexRetriever(
            index=vector_index, similarity_top_k=self.similarity_top_k
        )
        bm25_retriever = BM25Retriever.from_defaults(
            nodes=chunks, similarity_top_k=self.similarity_top_k
        )
        return QueryFusionRetriever(
            retrievers=[vector_retriever, bm25_retriever],
            similarity_top_k=self.similarity_top_k,
            num_queries=self.num_queries,
            mode="reciprocal_rerank",
            use_async=False,
            llm=self.llm,
        )

    # ── Classification ─────────────────────────────────────────────────────────

    @staticmethod
    def _parse_category(response_text: str) -> str:
        normalised = response_text.strip().lower()
        for cat in _PHARMA_DOC_CATEGORIES:
            if cat != "unknown" and cat in normalised:
                return cat
        return "unknown"

    def _classify_query(self, query: str) -> str:
        categories_str = ", ".join(_PHARMA_DOC_CATEGORIES)
        prompt = (
            "You are an expert pharmaceutical document classifier.\n"
            "Given a user query, identify which document type most likely contains the answer.\n"
            f"Choose exactly one from: {categories_str}.\n"
            "Respond with only the category name in snake_case. No extra text.\n\n"
            f"Query: {query}\n"
            "Category:"
        )
        response = self.llm.complete(prompt)
        return self._parse_category(response.text)

    def _classify_documents_batch(self, page_texts: List[str], batch_size: int = 5) -> List[str]:
        categories_str = ", ".join(_PHARMA_DOC_CATEGORIES)
        results: List[str] = []
        for batch_start in range(0, len(page_texts), batch_size):
            batch = page_texts[batch_start: batch_start + batch_size]
            pages_block = ""
            for i, text in enumerate(batch, 1):
                snippet = text[:_DOC_CLASSIFY_PROMPT_CHARS]
                pages_block += f"[Page {i}]\n{snippet}\n\n"
            prompt = (
                "You are an expert pharmaceutical document classifier.\n"
                f"Classify each page below. Choose exactly one from: {categories_str}.\n"
                "Respond with ONLY the category names, one per line. No extra text.\n\n"
                "Examples:\n"
                f"{_FEW_SHOT_EXAMPLES}"
                f"{pages_block}"
                "Categories (one per line):"
            )
            response = self.llm.complete(prompt)
            lines = [ln.strip() for ln in response.text.strip().splitlines() if ln.strip()]
            for idx in range(len(batch)):
                raw = lines[idx] if idx < len(lines) else ""
                results.append(self._parse_category(raw))
        return results

    def _annotate_pharma_doc_types(self, documents: List[Document]) -> List[Document]:
        needs_llm: List[int] = []
        for i, doc in enumerate(documents):
            header = doc.text[:300].lower()
            matched = next(
                (cat for cat, kws in _KEYWORD_MAP.items() if any(kw in header for kw in kws)),
                None,
            )
            if matched:
                doc.metadata["pharma_doc_type"] = matched
            else:
                needs_llm.append(i)
        if needs_llm:
            texts = [documents[i].text for i in needs_llm]
            llm_labels = self._classify_documents_batch(texts)
            for doc_idx, label in zip(needs_llm, llm_labels):
                documents[doc_idx].metadata["pharma_doc_type"] = label
        return documents

    def _build_filtered_engine(self, pharma_doc_type: str) -> RetrieverQueryEngine:
        if self._vector_index is None:
            raise RuntimeError("Pipeline not built.")
        filters = MetadataFilters(
            filters=[MetadataFilter(key="pharma_doc_type", value=pharma_doc_type)]
        )
        vector_retriever = VectorIndexRetriever(
            index=self._vector_index,
            similarity_top_k=self.similarity_top_k,
            filters=filters,
        )
        filtered_chunks = [
            c for c in self._chunks if c.metadata.get("pharma_doc_type") == pharma_doc_type
        ]
        if filtered_chunks:
            bm25_retriever = BM25Retriever.from_defaults(
                nodes=filtered_chunks,
                similarity_top_k=min(self.similarity_top_k, len(filtered_chunks)),
            )
            retrievers: List[Any] = [vector_retriever, bm25_retriever]
        else:
            retrievers = [vector_retriever]
        hybrid_retriever = QueryFusionRetriever(
            retrievers=retrievers,
            similarity_top_k=self.similarity_top_k,
            num_queries=self.num_queries,
            mode="reciprocal_rerank",
            use_async=False,
            llm=self.llm,
        )
        return RetrieverQueryEngine.from_args(
            retriever=hybrid_retriever,
            llm=self.llm,
            text_qa_template=_PHARMA_QA_PROMPT,
        )

    # ── Public API ─────────────────────────────────────────────────────────────

    def build(self, pdf_path: str, classify_docs: bool = False) -> None:
        self._pdf_path = pdf_path
        stored_classifications: Dict[tuple, str] = {}
        if self.persist_dir and os.path.exists(self.persist_dir):
            try:
                storage_context = StorageContext.from_defaults(persist_dir=self.persist_dir)
                stored_index = load_index_from_storage(storage_context)
                for chunk in stored_index.docstore.docs.values():
                    meta = getattr(chunk, "metadata", {})
                    fn = meta.get("file_name")
                    pg = meta.get("page_number")
                    dt = meta.get("pharma_doc_type")
                    if fn is not None and pg is not None and dt:
                        stored_classifications[(fn, pg)] = dt
            except Exception as e:
                logger.warning("Failed to load stored classifications: %s", e)

        documents = self.load_pdf(pdf_path)
        if classify_docs:
            if stored_classifications:
                needs_llm = []
                for doc in documents:
                    stored_type = stored_classifications.get(
                        (doc.metadata.get("file_name"), doc.metadata.get("page_number"))
                    )
                    if stored_type:
                        doc.metadata["pharma_doc_type"] = stored_type
                    else:
                        needs_llm.append(doc)
                if needs_llm:
                    llm_labels = self._classify_documents_batch([d.text for d in needs_llm])
                    for doc, label in zip(needs_llm, llm_labels):
                        doc.metadata["pharma_doc_type"] = label
            else:
                documents = self._annotate_pharma_doc_types(documents)
            before = len(documents)
            documents = [d for d in documents if d.metadata.get("pharma_doc_type") != "unknown"]
            dropped = before - len(documents)
            if dropped:
                logger.info("Skipping %d page(s) classified as unknown.", dropped)
        self._docs_classified = classify_docs
        self._chunks = self._chunk(documents)
        self._vector_index = self._index(self._chunks)
        hybrid_retriever = self._build_retriever(self._vector_index, self._chunks)
        self._query_engine = RetrieverQueryEngine.from_args(
            retriever=hybrid_retriever,
            llm=self.llm,
            text_qa_template=_PHARMA_QA_PROMPT,
        )
        self.clear_cache()
        logger.info("RAG pipeline ready.")

    def build_from_multiple_pdfs(
        self,
        pdf_paths: List[str],
        classify_docs: bool = False,
        progress_callback=None,
    ) -> None:
        if not pdf_paths:
            raise ValueError("No PDF paths provided.")
        if self.persist_dir and os.path.exists(self.persist_dir):
            shutil.rmtree(self.persist_dir)
        all_documents: List[Document] = []
        for idx, pdf_path in enumerate(pdf_paths, 1):
            docs = self.load_pdf(pdf_path)
            all_documents.extend(docs)
            if progress_callback:
                progress_callback(idx, len(pdf_paths), os.path.basename(pdf_path))
        if classify_docs:
            all_documents = self._annotate_pharma_doc_types(all_documents)
            before = len(all_documents)
            all_documents = [d for d in all_documents if d.metadata.get("pharma_doc_type") != "unknown"]
            dropped = before - len(all_documents)
            if dropped:
                logger.info("Skipping %d page(s) classified as unknown.", dropped)
        self._docs_classified = classify_docs
        self._chunks = self._chunk(all_documents)
        self._vector_index = self._index(self._chunks)
        hybrid_retriever = self._build_retriever(self._vector_index, self._chunks)
        self._query_engine = RetrieverQueryEngine.from_args(
            retriever=hybrid_retriever,
            llm=self.llm,
            text_qa_template=_PHARMA_QA_PROMPT,
        )
        self._pdf_path = f"Multiple files ({len(pdf_paths)} PDFs)"
        self.clear_cache()
        logger.info("RAG pipeline ready with %d documents.", len(pdf_paths))

    def clear_cache(self) -> None:
        self._query_cache.clear()

    def get_stats(self) -> Dict[str, Any]:
        if not self._chunks:
            raise RuntimeError("Pipeline not built.")
        file_names = set()
        page_types: Dict[tuple, str] = {}
        for chunk in self._chunks:
            fn = chunk.metadata.get("file_name")
            if fn:
                file_names.add(fn)
            pg = chunk.metadata.get("page_number")
            dt = chunk.metadata.get("pharma_doc_type", "unclassified")
            if fn is not None and pg is not None:
                page_types[(fn, pg)] = dt
        doc_type_counts: Dict[str, int] = {}
        for dt in page_types.values():
            doc_type_counts[dt] = doc_type_counts.get(dt, 0) + 1
        return {
            "total_pages": len(page_types),
            "total_chunks": len(self._chunks),
            "total_files": len(file_names),
            "file_names": sorted(list(file_names)),
            "doc_type_counts": doc_type_counts,
            "classified": self._docs_classified,
        }

    def get_document_details(self) -> List[Dict[str, Any]]:
        if not self._chunks:
            raise RuntimeError("Pipeline not built.")
        file_data: Dict[str, Dict[str, Any]] = {}
        for chunk in self._chunks:
            meta = chunk.metadata
            fname = meta.get("file_name", "unknown")
            page = meta.get("page_number")
            ocr_used = meta.get("ocr_used", False)
            doc_type = meta.get("doc_type", "digital")
            pharma_type = meta.get("pharma_doc_type")
            if fname not in file_data:
                file_data[fname] = {
                    "file_name": fname, "pages": set(), "scanned_pages": set(),
                    "total_chunks": 0, "has_ocr": False, "pharma_page_types": {},
                }
            fd = file_data[fname]
            fd["total_chunks"] += 1
            if page is not None:
                fd["pages"].add(page)
                if ocr_used or doc_type == "scanned":
                    fd["scanned_pages"].add(page)
            if ocr_used:
                fd["has_ocr"] = True
            if pharma_type and page is not None:
                fd["pharma_page_types"].setdefault(pharma_type, set()).add(page)
        result = []
        for fname, fd in sorted(file_data.items()):
            total_pages = len(fd["pages"])
            scanned = len(fd["scanned_pages"])
            result.append({
                "file_name": fname,
                "total_pages": total_pages,
                "total_chunks": fd["total_chunks"],
                "has_ocr": fd["has_ocr"],
                "scan_ratio": round(scanned / total_pages, 2) if total_pages else 0.0,
                "pharma_doc_types": {k: len(v) for k, v in fd["pharma_page_types"].items()},
            })
        return result

    def expand_query(self, query: str, num_expansions: int = 3) -> List[str]:
        prompt = (
            f"I need to search a document with this query: \"{query}\"\n\n"
            f"Please generate {num_expansions} alternative versions that:\n"
            "1. Use different but related terminology\n"
            "2. Include relevant pharmaceutical/quality terms\n"
            "3. Cover similar concepts but phrased differently\n\n"
            "Format your response as a list of alternative queries only, with no additional text."
        )
        response = self.llm.complete(prompt)
        expansions = [line.strip() for line in response.text.split("\n") if line.strip()]
        if query not in expansions:
            expansions = [query] + expansions
        return expansions

    def query(
        self, question: str, expand: bool = False, num_expansions: int = 3, classify: bool = False
    ) -> str:
        if self._query_engine is None:
            raise RuntimeError("Pipeline not built.")
        search_query = question
        if expand:
            queries = self.expand_query(question, num_expansions=num_expansions)
            search_query = queries[1] if len(queries) > 1 else question
        engine = self._query_engine
        if classify and self._docs_classified:
            query_category = self._classify_query(search_query)
            if query_category != "unknown":
                engine = self._build_filtered_engine(query_category)
        response = engine.query(search_query)
        return str(response)

    def query_with_sources(
        self, question: str, expand: bool = False, num_expansions: int = 3, classify: bool = False
    ) -> Dict[str, Any]:
        if self._query_engine is None:
            raise RuntimeError("Pipeline not built.")
        cache_key = (question, expand, num_expansions, classify, self.similarity_top_k)
        if cache_key in self._query_cache:
            return self._query_cache[cache_key]
        search_query = question
        if expand:
            queries = self.expand_query(question, num_expansions=num_expansions)
            search_query = queries[1] if len(queries) > 1 else question
        query_category: Optional[str] = None
        engine = self._query_engine
        if classify:
            query_category = self._classify_query(search_query)
            if self._docs_classified and query_category != "unknown":
                engine = self._build_filtered_engine(query_category)
        response = engine.query(search_query)
        raw_scores = [n.score for n in response.source_nodes if n.score is not None]
        max_score = max(raw_scores) if raw_scores else 0.0
        sources: List[Dict[str, Any]] = []
        for rank, node in enumerate(response.source_nodes):
            meta = node.node.metadata
            confidence = (
                round((node.score / max_score) * 100, 1)
                if max_score > 0 and node.score is not None
                else round(100.0 / (rank + 1), 1)
            )
            sources.append({
                "text": node.node.text,
                "file": meta.get("file_name", "unknown"),
                "page": meta.get("page_number", "?"),
                "score": confidence,
                "doc_type": meta.get("doc_type", "digital"),
                "pharma_doc_type": meta.get("pharma_doc_type", "unknown"),
            })
        result = {
            "answer": str(response), "sources": sources,
            "chunk_count": len(sources), "query_category": query_category,
        }
        self._query_cache[cache_key] = result
        return result

    def stream_query_with_sources(
        self, question: str, expand: bool = False, num_expansions: int = 3, classify: bool = False
    ) -> "Iterator[Dict[str, Any]]":
        if self._query_engine is None:
            raise RuntimeError("Pipeline not built.")
        search_query = question
        if expand:
            queries = self.expand_query(question, num_expansions=num_expansions)
            search_query = queries[1] if len(queries) > 1 else question
        query_category: Optional[str] = None
        if classify:
            query_category = self._classify_query(search_query)
        retriever = (
            self._build_filtered_engine(query_category).retriever
            if classify and self._docs_classified and query_category and query_category != "unknown"
            else self._query_engine.retriever
        )
        source_nodes = retriever.retrieve(search_query)
        raw_scores = [n.score for n in source_nodes if n.score is not None]
        max_score = max(raw_scores) if raw_scores else 0.0
        sources: List[Dict[str, Any]] = []
        context_parts: List[str] = []
        for rank, node in enumerate(source_nodes):
            meta = node.node.metadata
            confidence = (
                round((node.score / max_score) * 100, 1)
                if max_score > 0 and node.score is not None
                else round(100.0 / (rank + 1), 1)
            )
            sources.append({
                "text": node.node.text,
                "file": meta.get("file_name", "unknown"),
                "page": meta.get("page_number", "?"),
                "score": confidence,
                "doc_type": meta.get("doc_type", "digital"),
                "pharma_doc_type": meta.get("pharma_doc_type", "unknown"),
            })
            context_parts.append(node.node.text)
        context_str = "\n\n".join(context_parts)
        prompt = _PHARMA_QA_PROMPT.format(context_str=context_str, query_str=search_query)
        prev_len = 0
        last_chunk = None
        for chunk in self.llm.stream_complete(prompt):
            current_text = chunk.text or ""
            token = current_text[prev_len:]
            prev_len = len(current_text)
            last_chunk = chunk
            yield {"token": token, "sources": None}
        full_answer = (last_chunk.text or "") if last_chunk else ""
        result = {
            "token": None, "answer": full_answer,
            "sources": sources, "chunk_count": len(sources), "query_category": query_category,
        }
        cache_key = (question, expand, num_expansions, classify, self.similarity_top_k)
        self._query_cache[cache_key] = {
            "answer": full_answer, "sources": sources,
            "chunk_count": len(sources), "query_category": query_category,
        }
        yield result


print("RAGPipeline class defined.")


In [ ]:
# ── Step 6: Initialize the pipeline ──────────────────────────────────────────
# MODEL_PATH was set in Step 4. If you resumed this runtime and lost the
# variable, re-run Step 4 first to restore it.

_pipeline_ready = False

rag = RAGPipeline(
    model_path=MODEL_PATH,
    persist_dir="/content/storage",
)
_pipeline_ready = rag._query_engine is not None
print("RAGPipeline initialized.")


In [ ]:
# ── Step 7: Gradio UI ─────────────────────────────────────────────────────────
# Launches a Gradio app with a public share URL.
# Upload one or more pharmaceutical PDFs, click "Build Index", then ask questions.
import gradio as gr

_PHARMA_LABELS = {
    "cover_letter": "Cover Letter",
    "certificate_of_quality": "Certificate of Quality",
    "packaging_specification": "Packaging Specification",
    "bse_tse_declaration": "BSE / TSE Declaration",
    "material_description": "Material Description",
    "supplier_qualification": "Supplier Qualification",
    "chain_of_custody": "Chain of Custody",
    "unknown": "Unclassified",
    "unclassified": "Unclassified",
}


def _scan_label(ratio: float) -> str:
    if ratio <= 0:
        return "Digital"
    if ratio < 0.5:
        return f"Mostly digital ({ratio:.0%} scanned)"
    if ratio < 1.0:
        return f"Mostly scanned ({ratio:.0%} scanned)"
    return "Fully scanned (OCR)"


def _format_document_panel(details: list) -> str:
    if not details:
        return "*No documents indexed yet.*"
    lines = [f"**{len(details)} document(s) in index**\n"]
    for doc in details:
        fname = doc["file_name"]
        lines.append(f"### {fname}\n")
        lines.append("| Property | Value |")
        lines.append("|---|---|")
        lines.append(f"| Pages indexed | {doc['total_pages']} |")
        lines.append(f"| Text chunks | {doc['total_chunks']} |")
        lines.append(f"| Format | {_scan_label(doc['scan_ratio'])} |")
        lines.append(f"| OCR applied | {'Yes' if doc['has_ocr'] else 'No'} |")
        pharma_types = doc["pharma_doc_types"]
        if pharma_types:
            sorted_types = sorted(pharma_types.items(), key=lambda item: -item[1])
            type_str = ", ".join(
                f"{_PHARMA_LABELS.get(k, k)} ({v}p)" for k, v in sorted_types
            )
            lines.append(f"| Document categories | {type_str} |")
        else:
            lines.append("| Document categories | *(none detected)* |")
        lines.append("\n---\n")
    return "\n".join(lines)


def build_pipeline(pdf_files, accumulated_files):
    global _pipeline_ready
    if pdf_files is None:
        pdf_files = []
    if not isinstance(pdf_files, list):
        pdf_files = [pdf_files]
    new_paths = [f.name if hasattr(f, "name") else f for f in pdf_files if f is not None]
    existing_names = {os.path.basename(p) for p in accumulated_files}
    for path in new_paths:
        base = os.path.basename(path)
        if base not in existing_names:
            accumulated_files = accumulated_files + [path]
            existing_names.add(base)
    if not accumulated_files:
        return (
            "No files uploaded.",
            "*Upload one or more PDFs and click **Build Index**.*",
            "*No documents indexed yet.*",
            accumulated_files,
        )
    try:
        _pipeline_ready = False
        progress_updates = []

        def progress_callback(current, total, filename):
            progress_updates.append(f"[ok] Loaded {current}/{total}: {filename}")

        if len(accumulated_files) == 1:
            rag.build(accumulated_files[0], classify_docs=True)
            file_label = os.path.basename(accumulated_files[0])
        else:
            rag.build_from_multiple_pdfs(
                accumulated_files, classify_docs=True, progress_callback=progress_callback,
            )
            file_label = f"{len(accumulated_files)} files"

        _pipeline_ready = True
        stats = rag.get_stats()
        stats_md = (
            f"| Stat | Value |\n|---|---|\n"
            f"| Files | {stats.get('total_files', 1)} |\n"
            f"| Pages | {stats['total_pages']} |\n"
            f"| Chunks | {stats['total_chunks']} |\n"
        )
        if stats.get("classified") and stats.get("doc_type_counts"):
            type_rows = "\n".join(
                f"| &nbsp;&nbsp;`{_PHARMA_LABELS.get(dt, dt)}` | {cnt} pages |"
                for dt, cnt in sorted(stats["doc_type_counts"].items())
            )
            stats_md += f"| **Document categories** | |\n{type_rows}\n"
        else:
            stats_md += "| Document categories | *(none detected)* |\n"
        if progress_updates:
            stats_md += "\n\n**Processing log:**\n\n"
            stats_md += "\n".join(f"- {e}" for e in progress_updates)
        docs_md = _format_document_panel(rag.get_document_details())
        return f"Ready - {file_label} (classified)", stats_md, docs_md, accumulated_files
    except Exception as exc:
        import traceback
        _pipeline_ready = False
        return (
            f"Error: {exc}",
            f"```\n{traceback.format_exc()}\n```",
            "*Error building document index.*",
            accumulated_files,
        )


def clear_files():
    global _pipeline_ready
    _pipeline_ready = False
    return (
        [],
        "No document loaded.",
        "*Stats will appear here after building the index.*",
        "*No documents indexed yet.*",
    )


def ask(question, history, classify, expand, num_expansions, top_k, filter_doc_type):
    if not question.strip():
        yield history, "", "*Ask a question above.*", ""
        return
    if not _pipeline_ready:
        yield (
            history + [
                {"role": "user", "content": question},
                {"role": "assistant", "content": "Please upload and build a document index first."},
            ],
            "", "*No sources - pipeline not ready.*", "",
        )
        return
    original_top_k = rag.similarity_top_k
    rag.similarity_top_k = top_k
    streaming_history = history + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": "_Thinking..._"},
    ]
    yield streaming_history, "", "*Retrieving sources...*", "Buffering response..."
    final_payload = None
    first_token_received = False
    try:
        for chunk in rag.stream_query_with_sources(
            question, classify=classify, expand=expand, num_expansions=num_expansions,
        ):
            token = chunk.get("token")
            if token is not None:
                if not first_token_received:
                    streaming_history[-1]["content"] = ""
                    first_token_received = True
                streaming_history[-1]["content"] += token
                yield streaming_history, "", "*Retrieving sources...*", ""
            else:
                final_payload = chunk
    except Exception as exc:
        rag.similarity_top_k = original_top_k
        yield (
            history + [
                {"role": "user", "content": question},
                {"role": "assistant", "content": f"Error: {exc}"},
            ],
            "", "*Error during retrieval.*", "",
        )
        return
    rag.similarity_top_k = original_top_k
    if final_payload is None:
        yield streaming_history, "", "*No response received.*", ""
        return
    if not streaming_history[-1]["content"].strip():
        streaming_history[-1]["content"] = (
            "_The model returned an empty response. "
            "This can happen on the first inference call - please ask again._"
        )
        yield streaming_history, "", "*No answer generated - try again.*", ""
        return
    sources = final_payload["sources"]
    query_category = final_payload["query_category"]
    if filter_doc_type and filter_doc_type != "all":
        sources = [s for s in sources if s.get("pharma_doc_type") == filter_doc_type]
        if not sources:
            yield (
                streaming_history, "",
                f"*No sources found for document type: {filter_doc_type}*", "",
            )
            return
    sources_md = f"**{len(sources)} chunk(s) retrieved**"
    if query_category:
        query_label = _PHARMA_LABELS.get(query_category, query_category)
        sources_md += f" &nbsp;-&nbsp; Query classified as: **`{query_label}`**"
    if filter_doc_type and filter_doc_type != "all":
        filter_label = _PHARMA_LABELS.get(filter_doc_type, filter_doc_type)
        sources_md += f" &nbsp;-&nbsp; Filtered to: **`{filter_label}`**"
    sources_md += "\n\n---\n\n"
    for index, source in enumerate(sources, 1):
        confidence = source["score"]
        confidence_str = f"{confidence:.1f}%" if confidence is not None else "N/A"
        pharma_label = _PHARMA_LABELS.get(source.get("pharma_doc_type", "unknown"), "Unclassified")
        scan_label = _scan_label(1.0 if source["doc_type"] == "scanned" else 0.0)
        snippet = source["text"][:300]
        ellipsis = "..." if len(source["text"]) > 300 else ""
        sources_md += (
            f"**Source {index}** &nbsp;-&nbsp; "
            f"`{source['file']}` &nbsp;-&nbsp; "
            f"Page **{source['page']}** &nbsp;-&nbsp; "
            f"Confidence: **{confidence_str}** &nbsp;-&nbsp; "
            f"{scan_label} &nbsp;-&nbsp; "
            f"**{pharma_label}**\n\n"
            f"> {snippet}{ellipsis}\n\n"
            f"---\n\n"
        )
    yield streaming_history, "", sources_md, ""


PFIZER_CSS = """
:root {
    --pf-bg: #f7f8fb; --pf-surface: #ffffff; --pf-surface-alt: #f3f6fa;
    --pf-border: #d9e1ea; --pf-text: #172132; --pf-muted: #526073;
    --pf-primary: #005eb8; --pf-primary-strong: #004b92;
    --pf-danger: #d4334f; --pf-radius: 10px;
}
.gradio-container {
    font-family: 'Source Sans 3', 'IBM Plex Sans', 'Segoe UI', sans-serif;
    background: var(--pf-bg); max-width: 1280px; color: var(--pf-text);
}
.pf-header {
    background: var(--pf-surface); border: 1px solid var(--pf-border);
    border-left: 5px solid var(--pf-primary); border-radius: var(--pf-radius);
    padding: 16px 20px; margin-bottom: 16px;
    display: flex; align-items: center; justify-content: space-between;
}
.pf-header-title { color: var(--pf-text); font-size: 1.25rem; font-weight: 700; margin: 0; }
.pf-header-sub   { color: var(--pf-muted); font-size: 0.85rem; margin: 3px 0 0; }
.pf-badge {
    background: var(--pf-surface-alt); color: var(--pf-primary-strong);
    border: 1px solid var(--pf-border); padding: 4px 10px; border-radius: 999px;
    font-size: 0.7rem; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase;
}
.pf-section {
    font-size: 0.68rem; font-weight: 700; letter-spacing: 0.12em; text-transform: uppercase;
    color: var(--pf-muted); margin: 0 0 10px; padding-bottom: 6px;
    border-bottom: 1px solid var(--pf-border);
}
.gradio-container .block, .gradio-container .gr-box,
.gradio-container .gr-panel, .gradio-container .gr-accordion,
.gradio-container details {
    background: var(--pf-surface); border: 1px solid var(--pf-border);
    border-radius: var(--pf-radius); color: var(--pf-text); box-shadow: none;
}
button.lg.primary, button.primary {
    background: var(--pf-primary); border: 1px solid var(--pf-primary);
    color: #ffffff; border-radius: 8px; font-weight: 600;
}
button.lg.primary:hover, button.primary:hover {
    background: var(--pf-primary-strong); border-color: var(--pf-primary-strong);
}
button.stop, button.secondary {
    background: var(--pf-surface); border: 1px solid var(--pf-border);
    color: var(--pf-text); border-radius: 8px;
}
textarea, input[type="text"], .gradio-container select {
    background: var(--pf-surface); color: var(--pf-text);
    border: 1px solid var(--pf-border); border-radius: 8px;
}
textarea::placeholder, input[type="text"]::placeholder { color: var(--pf-muted); }
#pf-chatbot {
    border: 1px solid var(--pf-border); border-radius: var(--pf-radius);
    background: var(--pf-surface);
}
#pf-chatbot .message-row.user-row .message, #pf-chatbot .user-row .message {
    background: var(--pf-primary); color: #ffffff; border-radius: 12px 12px 2px 12px;
}
#pf-chatbot .message-row.bot-row .message, #pf-chatbot .bot-row .message {
    background: #121212; border: 1px solid #202020;
    color: #ffffff; border-radius: 12px 12px 12px 2px;
}
#pf-chatbot .message-row .message, #pf-chatbot .message-row .message *,
#pf-chatbot .message-row .message p, #pf-chatbot .message-row .message span,
#pf-chatbot .message-row .message strong, #pf-chatbot .message-row .message em,
#pf-chatbot .message-row .message code { color: #ffffff !important; }
.status-box textarea {
    color: var(--pf-text); background: var(--pf-surface-alt); border-color: var(--pf-border);
}
.gradio-container .prose { color: var(--pf-text) !important; }
.gradio-container .prose p, .gradio-container .prose li,
.gradio-container .prose span { color: var(--pf-text) !important; }
.gradio-container .prose strong { color: var(--pf-text) !important; font-weight: 700; }
.gradio-container .prose em { color: var(--pf-muted) !important; }
.gradio-container .prose code {
    background: #e8f0f8 !important; color: var(--pf-primary-strong) !important;
    padding: 2px 6px; border-radius: 4px; font-size: 0.9em;
}
.gradio-container .prose blockquote {
    border-left: 3px solid var(--pf-border); color: var(--pf-text) !important;
    font-style: italic; background: var(--pf-surface-alt); padding: 8px 12px; margin: 8px 0;
}
.gradio-container .prose h1, .gradio-container .prose h2,
.gradio-container .prose h3, .gradio-container .prose h4 {
    color: var(--pf-text) !important; font-weight: 700;
}
.gradio-container .prose hr { border-color: var(--pf-border) !important; }
.gradio-container .prose table { border-collapse: collapse; width: 100%; }
.gradio-container .prose table th, .gradio-container .prose table td {
    background: var(--pf-text) !important; color: #ffffff !important;
    border: 1px solid var(--pf-border); padding: 8px 12px;
}
.gradio-container .prose table th { font-weight: 700; }
.gradio-container details > summary,
.gradio-container details > summary span,
.gradio-container details > summary button,
.gradio-container details > summary * { color: #000000 !important; }
.gradio-container label, .gradio-container label span,
.gradio-container .label-wrap, .gradio-container .label-wrap span,
.gradio-container .block-label, .gradio-container .block-label span { color: #000000 !important; }
.gradio-container .gr-slider label, .gradio-container .gr-slider label span,
.gradio-container .gr-dropdown label, .gradio-container .gr-dropdown label span,
.gradio-container .gr-checkbox label, .gradio-container .gr-checkbox label span,
.gradio-container .checkbox-label, .gradio-container .checkbox-label span { color: #000000 !important; }
.gradio-container .info, .gradio-container .info span,
.gradio-container .description, .gradio-container .description span { color: #000000 !important; }
.gradio-container .upload-container span, .gradio-container .upload-container .wrap span,
.gradio-container .file-drop-zone span, .gradio-container .file-upload-title,
.gradio-container [data-testid="file"] span, .gradio-container .upload span,
.gradio-container .upload-btn span { color: #000000 !important; }
#pf-chatbot .placeholder, #pf-chatbot .placeholder p,
#pf-chatbot .placeholder span, #pf-chatbot .placeholder *,
#pf-chatbot [data-testid="empty-state"],
#pf-chatbot [data-testid="empty-state"] p,
#pf-chatbot [data-testid="empty-state"] * { color: #ffffff !important; }
"""

_PFIZER_THEME = gr.themes.Base(
    primary_hue=gr.themes.colors.blue,
    neutral_hue=gr.themes.colors.gray,
    font=[
        gr.themes.GoogleFont("Source Sans 3"),
        gr.themes.GoogleFont("IBM Plex Sans"),
        "Segoe UI", "sans-serif",
    ],
).set(
    body_background_fill="#f7f8fb",
    body_text_color="#172132",
    body_text_color_subdued="#526073",
    block_background_fill="#ffffff",
    block_border_color="#d9e1ea",
    block_border_width="1px",
    block_radius="10px",
    block_shadow="0 0 0 rgba(0,0,0,0)",
    block_title_text_color="#172132",
    block_label_text_color="#172132",
    input_background_fill="#ffffff",
    input_border_color="#d9e1ea",
    input_border_color_focus="#005eb8",
    input_placeholder_color="#526073",
    button_primary_background_fill="#005eb8",
    button_primary_background_fill_hover="#004b92",
    button_primary_text_color="#ffffff",
    button_secondary_background_fill="#ffffff",
    button_secondary_border_color="#d9e1ea",
    button_secondary_text_color="#172132",
    checkbox_label_text_color="#172132",
    checkbox_label_text_color_selected="#172132",
    checkbox_background_color_selected="#005eb8",
)

with gr.Blocks(
    title="Pfizer | Pharmaceutical Document QA",
    css=PFIZER_CSS,
    theme=_PFIZER_THEME,
) as demo:
    gr.HTML("""
        <div class="pf-header">
            <div>
                <p class="pf-header-title">Pharmaceutical Document QA</p>
                <p class="pf-header-sub">
                    Upload pharmaceutical PDFs and ask questions - powered by hybrid RAG retrieval
                    with automatic document classification
                </p>
            </div>
            <span class="pf-badge">RAG System</span>
        </div>
    """)

    accumulated_files_state = gr.State([])

    with gr.Row(equal_height=False):
        with gr.Column(scale=2, min_width=300):
            gr.HTML('<p class="pf-section">Documents</p>')
            pdf_input = gr.File(label="Upload PDF(s)", file_types=[".pdf"], file_count="multiple")
            with gr.Row():
                build_btn = gr.Button("Build Index", variant="primary", size="lg", scale=3)
                clear_btn = gr.Button("Clear", variant="stop", size="sm", scale=1)
            with gr.Accordion("Document Details", open=True):
                docs_display = gr.Markdown("*No documents indexed yet.*")
            with gr.Accordion("Index Summary", open=False):
                stats_display = gr.Markdown("*Stats will appear here after building the index.*")

        with gr.Column(scale=3, min_width=420):
            gr.HTML('<p class="pf-section">Query</p>')
            chatbot = gr.Chatbot(
                height=440, show_label=False, elem_id="pf-chatbot",
                placeholder=(
                    "<div style='text-align:center;padding:40px 20px'>"
                    "<p style='font-size:1.1rem;font-weight:600;margin-bottom:6px;color:#ffffff'>"
                    "No conversation yet</p>"
                    "<p style='font-size:0.85rem;color:#ffffff'>"
                    "Build the index and ask a question to get started.</p></div>"
                ),
            )
            with gr.Row():
                question_input = gr.Textbox(
                    placeholder="e.g. What are the storage conditions for this batch?",
                    show_label=False, scale=5, lines=1, max_lines=4, container=False,
                )
                ask_btn = gr.Button("Ask", variant="primary", scale=1, min_width=72)

        with gr.Column(scale=2, min_width=320):
            gr.HTML('<p class="pf-section">Info</p>')
            status_box = gr.Textbox(
                label="Status",
                value="No document loaded." if not _pipeline_ready else "Ready - index loaded from disk.",
                interactive=False, max_lines=2, elem_classes=["status-box"],
            )
            response_loading_box = gr.Textbox(
                label="Response Status", value="",
                placeholder="Response buffering state appears here...",
                interactive=False, max_lines=1, elem_classes=["status-box"],
            )
            with gr.Accordion("Sources & Evidence", open=True):
                sources_display = gr.Markdown(
                    "*Sources and confidence scores will appear here after asking a question.*"
                )
            with gr.Accordion("Settings", open=False):
                gr.Markdown("**Retrieval**")
                top_k = gr.Slider(
                    minimum=1, maximum=20, value=5, step=1, label="Chunks to retrieve (top-k)",
                    info="Higher values provide more context but may slow inference.",
                )
                filter_doc_type = gr.Dropdown(
                    choices=[
                        ("All document types", "all"),
                        ("Cover Letter", "cover_letter"),
                        ("Certificate of Quality", "certificate_of_quality"),
                        ("Packaging Specification", "packaging_specification"),
                        ("BSE / TSE Declaration", "bse_tse_declaration"),
                        ("Material Description", "material_description"),
                        ("Supplier Qualification", "supplier_qualification"),
                        ("Chain of Custody", "chain_of_custody"),
                        ("Unclassified", "unknown"),
                    ],
                    value="all", label="Filter by document type",
                    info="Requires classification to be enabled during indexing.",
                )
                gr.Markdown("**Query Processing**")
                classify_query_toggle = gr.Checkbox(
                    label="Auto-classify query", value=False,
                    info="Uses the LLM to detect the most relevant document category.",
                )
                expand_query_toggle = gr.Checkbox(
                    label="Query expansion", value=False,
                    info="Generates alternative phrasings to improve retrieval recall.",
                )
                num_expansions = gr.Slider(
                    minimum=1, maximum=5, value=3, step=1, label="Number of expansions",
                    info="Only active when query expansion is enabled.",
                )

    build_btn.click(
        build_pipeline, inputs=[pdf_input, accumulated_files_state],
        outputs=[status_box, stats_display, docs_display, accumulated_files_state],
    )
    clear_btn.click(
        clear_files, inputs=[],
        outputs=[accumulated_files_state, status_box, stats_display, docs_display],
    )
    ask_btn.click(
        ask,
        inputs=[question_input, chatbot, classify_query_toggle, expand_query_toggle,
                num_expansions, top_k, filter_doc_type],
        outputs=[chatbot, question_input, sources_display, response_loading_box],
    )
    question_input.submit(
        ask,
        inputs=[question_input, chatbot, classify_query_toggle, expand_query_toggle,
                num_expansions, top_k, filter_doc_type],
        outputs=[chatbot, question_input, sources_display, response_loading_box],
    )

# share=True is required in Google Colab to generate a public tunnel URL
demo.launch(share=True)
